# LLM API Demo — PaathSaathi Case Study

**LLM server:** [Ollama](https://ollama.com) (`ollama serve`)

---

## The case

**PaathSaathi** helps first-generation students with **NSP (National Scholarship Portal)** queries over WhatsApp.

**Arjun** calls the LLM via HTTP, parses JSON, **validates with Pydantic**, and fills the counselor dashboard.

> Educational demo only — not real scholarship or legal advice.

---

## What you will learn

1. **API requests** to an LLM with `requests`
2. **Secure config** via `.env` (not hardcoded keys)
3. **Parse JSON** from HTTP responses
4. **Validate** LLM output with **Pydantic schemas**
5. **Extract** dashboard fields safely
6. **Debug** common API errors

**Same message every act** — student `STU-2847`.


In [ ]:
# ── Standard library + third-party imports ───────────────────────────────
# json      → parse text into Python dicts/lists
# os        → read environment variables (API keys, URLs)
# re        → strip ```json fences models sometimes add
# Enum      → restrict urgency to LOW | MEDIUM | HIGH
# Path      → locate the .env file next to this notebook
# requests  → send HTTP POST to Ollama
# dotenv    → load PAATHSAATHI_API_KEY from .env file
# pydantic  → validate LLM JSON against a strict schema

import json
import os
import re
from enum import Enum
from pathlib import Path

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError, field_validator

print("Imports OK — ready to call Ollama")


Imports OK — ready to call Ollama


In [ ]:
# ── Case constants (same values reused in EVERY act below) ───────────────
# In a real app these come from your database / WhatsApp webhook payload.

STUDENT_ID = "STU-2847"  # primary key shown on counselor dashboard

# The WhatsApp message we analyze throughout the notebook
CUSTOMER_MESSAGE = (
    "Namaste, main STU-2847 hoon, Class 12 PCM from Nashik. "
    "Mummy ki income Rs 2.8 lakh hai. Merit scholarship ke liye apply karna hai "
    "par NSP pe 'income certificate' reject ho gaya — 'name mismatch' likha aaya. "
    "Kya main dobara upload kar sakta hoon? Last date 12 tarikh hai. "
    "Aadhaar pe naam 'Rahul K' hai, certificate pe 'Rahul Kumar'. Kya eligible hoon?"
)

# Local Ollama model — small enough for classroom laptops
MODEL = "llama3.2:3b"

print("Student:", STUDENT_ID)
print("Message preview:", CUSTOMER_MESSAGE[:80], "…")


Student: STU-2847
Message preview: Namaste, main STU-2847 hoon, Class 12 PCM from Nashik. Mummy ki income Rs 2.8 la …


In [ ]:
# ── Load secrets from .env (never commit real keys to git) ───────────────
# Copy .env.example → .env and edit values there.
# load_dotenv() reads KEY=value lines into os.environ

load_dotenv(Path(".env"))

# Base URL of Ollama HTTP server (default local install)
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")

# Simulated app-layer API key — our wrapper checks this before calling Ollama
API_KEY = os.getenv("PAATHSAATHI_API_KEY", "")


print("Ollama URL:", OLLAMA_BASE_URL)
print("API key loaded:", "yes ✓" if API_KEY else "MISSING ✗ — copy .env.example to .env")


Ollama URL: http://localhost:11434
API key loaded: yes ✓


---

## Act 1 — Make an API request

Ollama exposes `POST /api/generate`. We send JSON: `model`, `prompt`, `stream: false`.


In [ ]:
# Step 1 — Build the HTTP request (we send it in the next cell)
# Ollama docs: POST /api/generate with JSON body

url = f"{OLLAMA_BASE_URL}/api/generate"

payload = {
    "model": MODEL,                    # which local model to run
    "prompt": (                        # text sent to the model
        f"Reply in one short sentence: what is the student's main worry?\n\n"
        f"{CUSTOMER_MESSAGE}"
    ),
    "stream": False,                   # False = one JSON response (easier for demos)
    "options": {
        "temperature": 0,              # 0 = deterministic answer
        "num_predict": 60,             # max output tokens
    },
}

print("URL:", url)
print("Payload keys:", list(payload.keys()))


URL: http://localhost:11434/api/generate
Payload keys: ['model', 'prompt', 'stream', 'options']


In [ ]:
# Step 2 — Send POST request and read HTTP status
# requests.post(..., json=payload) sets Content-Type: application/json automatically

response = requests.post(url, json=payload, timeout=120)

# 200 = success; 4xx/5xx = client/server error (see Act 6)
print("HTTP status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))


HTTP status: 200
Content-Type: application/json; charset=utf-8


In [ ]:
# Step 3 — Parse the HTTP body as JSON and inspect useful fields
# Ollama wraps model text in a JSON object — the answer is NOT the raw HTTP text

body = response.json()  # convert JSON string → Python dict

print("Top-level keys:", list(body.keys()))
print()
print("Model text (response field):")
print(body.get("response", "").strip())
print()
# Bonus metadata — useful for cost/latency debugging
print("Token counts — prompt:", body.get("prompt_eval_count"), "| output:", body.get("eval_count"))


Top-level keys: ['model', 'created_at', 'response', 'done', 'done_reason', 'context', 'total_duration', 'load_duration', 'prompt_eval_count', 'prompt_eval_duration', 'eval_count', 'eval_duration']

Model text (response field):
Mummy ki income certificate mein naam mismatch ho gaya hai, isliye aap dobara upload karne ke liye NSP ki website par check karein.

Token counts — prompt: 149 | output: 37


### Takeaway (Act 1)

`requests.post(url, json=payload)` → JSON body with **`response`** (model text) + metadata.


---

## Act 2 — Secure authentication

Store **`PAATHSAATHI_API_KEY`** in `.env`. Our wrapper rejects wrong keys before calling Ollama.


In [ ]:
# Reusable helper: check API key, then call Ollama
# In production YOUR backend would validate keys before proxying to an LLM provider

def call_llm(prompt: str, api_key: str | None, max_tokens: int = 120) -> requests.Response:
    # Compare caller's key with the one loaded from .env
    if api_key != os.getenv("PAATHSAATHI_API_KEY"):
        raise PermissionError("401 Unauthorized — invalid or missing PAATHSAATHI_API_KEY")

    # Same endpoint as Act 1 — now behind an auth gate
    return requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={
            "model": MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {"num_predict": max_tokens},
        },
        timeout=120,
    )

print("call_llm() defined — use API_KEY from .env for all later acts")


call_llm() defined — use API_KEY from .env for all later acts


In [ ]:
# Security demo — wrong key must be rejected BEFORE any network call to Ollama
# Never hardcode secrets like this in real code (shown here only to demo failure)

try:
    call_llm("test", api_key="wrong-key-on-purpose")
    print("Unexpected success — auth check failed!")
except PermissionError as e:
    print("✗ Blocked as expected:", e)


✗ Blocked as expected: 401 Unauthorized — invalid or missing PAATHSAATHI_API_KEY


In [ ]:
# Correct key from .env → request proceeds to Ollama
r = call_llm("Say OK in one word.", api_key=API_KEY, max_tokens=5)

print("✓ HTTP", r.status_code)
print("Response JSON keys:", list(r.json().keys())[:4])


✓ HTTP 200
Response JSON keys: ['model', 'created_at', 'response', 'done']


---

## Act 3 — Parse JSON responses

Two layers: (1) HTTP body from Ollama, (2) JSON **inside** the model's text.


In [ ]:
# Helpers for Layer 2 — model often wraps JSON in markdown code fences

def strip_markdown_fence(text: str) -> str:
    """Remove ```json ... ``` if the model adds markdown formatting."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.MULTILINE).strip()
    return text


def parse_inner_json(text: str) -> dict:
    """Parse JSON embedded inside the model's text response."""
    return json.loads(strip_markdown_fence(text))

print("JSON helpers defined — used whenever the model returns JSON as text")


JSON helpers defined — used whenever the model returns JSON as text


In [ ]:
# Ask the model to return a SMALL JSON object inside its text field
# We will parse two layers: HTTP JSON (Ollama) → inner JSON (model output)

prompt = (
    "Analyze this message. Reply JSON only: "
    '{\"intent\": \"\", \"urgency\": \"LOW|MEDIUM|HIGH\"}\n\n'
    + CUSTOMER_MESSAGE
)

r = call_llm(prompt, api_key=API_KEY, max_tokens=120)
http_data = r.json()  # Layer 1 — full Ollama response as dict


In [ ]:
# Layer 1 — metadata from Ollama (not the counselor dashboard fields yet)
print("Layer 1 — Ollama HTTP JSON fields:")
for key in ("model", "done", "prompt_eval_count", "eval_count"):
    print(f"  {key}: {http_data.get(key)}")


Layer 1 — Ollama HTTP JSON fields:
  model: llama3.2:3b
  done: True
  prompt_eval_count: 159
  eval_count: 15


In [ ]:
# Layer 2 — the model's answer lives in the 'response' string
model_text = http_data["response"].strip()

print("Layer 2 — raw text from model:")
print(model_text)
print()

# Second json.loads — because the model returned JSON *as text*
try:
    inner = parse_inner_json(model_text)
    print("Parsed inner dict:", inner)
except json.JSONDecodeError as e:
    # Common when model adds extra prose outside JSON
    print("JSONDecodeError:", e)


Layer 2 — raw text from model:
{"intent": "HELP", "urgency": "MEDIUM"}

Parsed inner dict: {'intent': 'HELP', 'urgency': 'MEDIUM'}


---

## Act 4 — Pydantic schema validation

**Problem:** LLMs return JSON-ish text — fields may be missing, wrong types, or invalid values.

**Solution:** Define a **Pydantic model** = your contract. `model_validate()` accepts good data and raises **`ValidationError`** on bad data.


In [ ]:
# Urgency is restricted to three values — invalid strings are rejected

class Urgency(str, Enum):
    LOW = "LOW"
    MEDIUM = "MEDIUM"
    HIGH = "HIGH"


class LLMExtraction(BaseModel):
    """
    Contract for what the LLM must return before we show data on the dashboard.
    Pydantic checks types, required fields, and custom rules automatically.
    """
    intent: str = Field(min_length=1, description="What the student wants")
    scheme_guess: str = Field(default="", description="Scholarship scheme if identifiable")
    missing_documents: list[str] = Field(default_factory=list)
    urgency: Urgency
    counselor_reply_draft: str = Field(min_length=10, description="Short Hinglish reply draft")

    @field_validator("counselor_reply_draft", mode="before")
    @classmethod
    def flatten_reply(cls, v):
        # Sometimes the model returns {"message": "..."} instead of a plain string
        if isinstance(v, dict):
            return v.get("message", str(v))
        return v

    @field_validator("urgency", mode="before")
    @classmethod
    def normalize_urgency(cls, v):
        # Accept "high" or " HIGH " and normalize to enum value
        if isinstance(v, str):
            return v.strip().upper()
        return v

print("LLMExtraction schema defined")


LLMExtraction schema defined


In [ ]:
# Export JSON Schema — same contract frontend / prompt docs can reference
schema = LLMExtraction.model_json_schema()

print("JSON Schema title:", schema.get("title"))
print("Required fields:", schema.get("required"))
print("Urgency allowed values:", [u.value for u in Urgency])


JSON Schema title: LLMExtraction
Required fields: ['intent', 'urgency', 'counselor_reply_draft']
Urgency allowed values: ['LOW', 'MEDIUM', 'HIGH']


In [ ]:
# ── Happy path: manually crafted valid dict passes validation ───────────

sample_valid = {
    "intent": "Fix income certificate name mismatch",
    "scheme_guess": "Merit scholarship",
    "missing_documents": ["income_certificate"],
    "urgency": "HIGH",
    "counselor_reply_draft": (
        "Namaste, name mismatch fix karke dubara upload karein. "
        "Deadline se pehle help karenge."
    ),
}

# model_validate converts dict → typed LLMExtraction object
validated_sample = LLMExtraction.model_validate(sample_valid)

print("✓ Valid sample passed")
print(validated_sample.model_dump())  # back to plain dict for JSON APIs


✓ Valid sample passed
{'intent': 'Fix income certificate name mismatch', 'scheme_guess': 'Merit scholarship', 'missing_documents': ['income_certificate'], 'urgency': <Urgency.HIGH: 'HIGH'>, 'counselor_reply_draft': 'Namaste, name mismatch fix karke dubara upload karein. Deadline se pehle help karenge.'}


In [ ]:
# ── Unhappy path: deliberately broken dict → ValidationError ────────────
# This mimics common LLM mistakes we must catch before updating the UI

sample_invalid = {
    "intent": "",                              # too short (min_length=1)
    "missing_documents": "income_certificate", # wrong type — should be list
    "urgency": "CRITICAL",                     # not in Urgency enum
    "counselor_reply_draft": "ok",             # too short (min_length=10)
}

try:
    LLMExtraction.model_validate(sample_invalid)
except ValidationError as e:
    print("✗ ValidationError (expected for bad LLM output):")
    print(e.errors()[0]["msg"], "→ field", e.errors()[0]["loc"])
    print(f"  ({len(e.errors())} total issues — fix prompt or retry LLM)")


✗ ValidationError (expected for bad LLM output):
String should have at least 1 character → field ('intent',)
  (4 total issues — fix prompt or retry LLM)


### Why Pydantic?

| Without schema | With Pydantic |
|----------------|---------------|
| Silent `UNKNOWN` defaults | Explicit **`ValidationError`** |
| Wrong types slip through | `list` vs `str` caught |
| Invalid urgency stored | **Enum** restricts values |


---

## Act 5 — Extract + validate live LLM output

Call Ollama → parse JSON → **`LLMExtraction.model_validate()`** → dashboard row.


In [ ]:
# Prompt includes an EXAMPLE JSON shape — helps the model fill all fields
extract_prompt = f"""You are PaathSaathi API. Analyze this WhatsApp message.
Reply with JSON only (no markdown). Example shape:
{{"intent":"Fix certificate name mismatch","scheme_guess":"Merit scholarship","missing_documents":["income_certificate"],"urgency":"HIGH","counselor_reply_draft":"Namaste, name mismatch ke liye corrected certificate dubara upload karein."}}

Message: {CUSTOMER_MESSAGE}"""

# Step 1 — get raw text from Ollama
r = call_llm(extract_prompt, api_key=API_KEY, max_tokens=280)
raw_text = r.json()["response"].strip()

print("Raw LLM text (what we must parse + validate):")
print(raw_text[:400], ("…" if len(raw_text) > 400 else ""))


Raw LLM text (what we must parse + validate):
{"intent":"Eligibility check","scheme_guess":"Merit scholarship","missing_documents":["income_certificate"],
"urgency":"MEDIUM","counselor_reply_draft":"Namaste STU-2847, aapki eligibility ki janch karna hai. Aapke aadhaar par naam 'Rahul K' aur certificate par 'Rahul Kumar' hain. Ise match nahi karne wale name mismatch ke liye aap dobara upload kar sakte hain. "} 


In [ ]:
# Step 2 — parse JSON string → Python dict (Layer 2 again)
try:
    raw_dict = parse_inner_json(raw_text)
    print("Parsed dict keys:", list(raw_dict.keys()))
except json.JSONDecodeError as e:
    print("JSONDecodeError — model did not return valid JSON:", e)
    raw_dict = {}


Parsed dict keys: ['intent', 'scheme_guess', 'missing_documents', 'urgency', 'counselor_reply_draft']


In [ ]:
# Step 3 — Pydantic gate: only validated data reaches the dashboard
try:
    extraction = LLMExtraction.model_validate(raw_dict)
    print("✓ Pydantic validation PASSED — safe to show counselors")
    print("  intent :", extraction.intent)
    print("  urgency:", extraction.urgency.value)
except ValidationError as e:
    extraction = None  # block bad data — do not render in UI
    print("✗ Pydantic validation FAILED — dashboard stays empty")
    for err in e.errors()[:3]:
        print(f"  field {err['loc']}: {err['msg']}")


✓ Pydantic validation PASSED — safe to show counselors
  intent : Eligibility check
  urgency: MEDIUM


In [ ]:
# Step 4 — map validated extraction → dashboard row (add student_id)

class DashboardRow(BaseModel):
    """Final shape stored in PaathSaathi database / UI."""
    student_id: str
    intent: str
    scheme_guess: str
    missing_documents: list[str]
    urgency: str
    counselor_reply_draft: str

    @classmethod
    def from_extraction(cls, student_id: str, ext: LLMExtraction) -> "DashboardRow":
        # Combine CRM id with validated LLM fields
        return cls(
            student_id=student_id,
            urgency=ext.urgency.value,
            **ext.model_dump(exclude={"urgency"}),
        )


if extraction:
    row = DashboardRow.from_extraction(STUDENT_ID, extraction)
    print("DASHBOARD ROW — PaathSaathi UI")
    print("=" * 56)
    for k, v in row.model_dump().items():
        print(f"  {k}: {v}")
else:
    print("Dashboard not updated — fix validation errors or adjust prompt first")


DASHBOARD ROW — PaathSaathi UI
  student_id: STU-2847
  intent: Eligibility check
  scheme_guess: Merit scholarship
  missing_documents: ['income_certificate']
  urgency: MEDIUM
  counselor_reply_draft: Namaste STU-2847, aapki eligibility ki janch karna hai. Aapke aadhaar par naam 'Rahul K' aur certificate par 'Rahul Kumar' hain. Ise match nahi karne wale name mismatch ke liye aap dobara upload kar sakte hain. 


---

## Act 6 — Debug common API errors

Same student case — learn to read errors quickly.


In [ ]:
# Small helper to print status or exception for debugging exercises

def try_request(label: str, url: str, payload: dict | None):
    print(f"--- {label} ---")
    try:
        resp = requests.post(url, json=payload, timeout=10)
        print("Status:", resp.status_code)
        print("Body:", (resp.text[:150] + "…") if len(resp.text) > 150 else resp.text)
    except requests.exceptions.ConnectionError as e:
        # Ollama not running OR wrong host/port
        print("ConnectionError:", e)
    except Exception as e:
        print(type(e).__name__ + ":", e)
    print()


In [ ]:
# Error 1 — typo in URL path → HTTP 404 Not Found
try_request(
    "Wrong path (404)",
    f"{OLLAMA_BASE_URL}/api/generatee",  # extra 'e' — common typo
    {"model": MODEL, "prompt": "hi", "stream": False},
)


--- Wrong path (404) ---
Status: 404
Body: 404 page not found



In [ ]:
# Error 2 — missing required 'model' field → Ollama returns error JSON
try_request(
    "Missing model field",
    f"{OLLAMA_BASE_URL}/api/generate",
    {"prompt": "hi", "stream": False},  # incomplete payload
)


--- Missing model field ---
Status: 404
Body: {"error":"model '' not found"}



In [ ]:
# Error 3 — KeyError when reading wrong JSON key
# Ollama puts model text in 'response', NOT 'answer' or 'content'

good = requests.post(
    f"{OLLAMA_BASE_URL}/api/generate",
    json={"model": MODEL, "prompt": "Say hi", "stream": False},
    timeout=60,
).json()

print("Correct: good.get('response') =", good.get("response", "")[:30])

try:
    _ = good["answer"]  # this key does not exist in Ollama's schema
except KeyError as e:
    print("KeyError:", e, "→ always check API docs for field names")


Correct: good.get('response') = Hi! How can I assist you today
KeyError: 'answer' → always check API docs for field names


In [ ]:
# Error 4 — connection refused (Ollama stopped or wrong port)
try_request(
    "Wrong port",
    "http://localhost:19999/api/generate",  # nothing listening here
    {"model": MODEL, "prompt": "hi", "stream": False},
)


--- Wrong port ---
ConnectionError: HTTPConnectionPool(host='localhost', port=19999): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=19999): Failed to establish a new connection: [Errno 61] Connection refused"))



---

## Journey — STU-2847

| Act | Skill | Key API / tool |
|-----|-------|----------------|
| 1 | HTTP request | `requests.post` → `/api/generate` |
| 2 | Secure config | `.env` + `getenv` |
| 3 | Parse JSON | `response.json()` + `json.loads` |
| 4 | **Pydantic schema** | `LLMExtraction`, `ValidationError` |
| 5 | Extract + validate | `model_validate()` → `DashboardRow` |
| 6 | Debug errors | 404, bad body, KeyError, connection |

---
